In [3]:
import pandas as pd
import numpy as np
import joblib

df_model = pd.read_csv("data/interim/train_cleaned_v1.csv")
X = df_model.drop("SeriousDlqin2yrs", axis=1)
y = df_model["SeriousDlqin2yrs"]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

xgb = joblib.load("models/xgboost_v1.pkl")
y_prob = xgb.predict_proba(X_test)[:, 1]

print("Test set size:", X_test.shape)
print("y_prob sample:", y_prob[:5])

Test set size: (29946, 10)
y_prob sample: [0.77109396 0.4781493  0.7914553  0.07603469 0.23549059]


In [4]:
# Cost assumption (stated explicitly, not measured from real financial data):
# Approving a defaulter (False Negative) costs COST_FN units.
# Rejecting a good borrower (False Positive) costs COST_FP units (lost profit margin).
# We assume a 5:1 ratio as a commonly-cited starting point in credit risk literature.
COST_FN = 5.0
COST_FP = 1.0

In [5]:
from sklearn.metrics import confusion_matrix

thresholds = np.arange(0.05, 0.95, 0.01)
results = []

for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    expected_loss = (fn * COST_FN) + (fp * COST_FP)
    results.append({"threshold": t, "fn": fn, "fp": fp, "expected_loss": expected_loss})

results_df = pd.DataFrame(results)
optimal_row = results_df.loc[results_df["expected_loss"].idxmin()]
print(optimal_row)

threshold           0.72
fn                907.00
fp               1971.00
expected_loss    6506.00
Name: 67, dtype: float64


In [6]:
for t in [0.1, 0.3, 0.5, 0.6, 0.72, 0.8, 0.9]:
    row = results_df.loc[(results_df["threshold"] - t).abs().idxmin()]
    print(row.to_dict())

{'threshold': 0.1, 'fn': 26.0, 'fp': 21822.0, 'expected_loss': 21952.0}
{'threshold': 0.30000000000000004, 'fn': 235.0, 'fp': 10395.0, 'expected_loss': 11570.0}
{'threshold': 0.5000000000000001, 'fn': 464.0, 'fp': 5613.0, 'expected_loss': 7933.0}
{'threshold': 0.6000000000000002, 'fn': 682.0, 'fp': 3628.0, 'expected_loss': 7038.0}
{'threshold': 0.7200000000000002, 'fn': 907.0, 'fp': 1971.0, 'expected_loss': 6506.0}
{'threshold': 0.8000000000000002, 'fn': 1094.0, 'fp': 1244.0, 'expected_loss': 6714.0}
{'threshold': 0.9000000000000002, 'fn': 1450.0, 'fp': 434.0, 'expected_loss': 7684.0}


In [7]:
import matplotlib.pyplot as plt
plt.plot(results_df["threshold"], results_df["expected_loss"])
plt.axvline(optimal_row["threshold"], color="red", linestyle="--", label="Optimal threshold")
plt.xlabel("Threshold")
plt.ylabel("Expected Loss (cost units)")
plt.title("Expected Loss vs Decision Threshold")
plt.legend()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [8]:
old_thresholds = {"Lead (0.5)": 0.5, "Balanced (0.6)": 0.6}

for name, t in old_thresholds.items():
    row = results_df.loc[(results_df["threshold"] - t).abs().idxmin()]
    old_loss = row["expected_loss"]
    optimal_loss = optimal_row["expected_loss"]
    pct_improvement = (old_loss - optimal_loss) / old_loss * 100
    print(f"{name}: expected_loss={old_loss:.0f} -> CostOptimized(0.72): {optimal_loss:.0f} "
          f"({pct_improvement:.1f}% reduction)")

Lead (0.5): expected_loss=7933 -> CostOptimized(0.72): 6506 (18.0% reduction)
Balanced (0.6): expected_loss=7038 -> CostOptimized(0.72): 6506 (7.6% reduction)


In [9]:
results_df.to_csv("models/threshold_expected_loss.csv", index=False)

In [10]:
import os
print(os.path.exists("models/threshold_expected_loss.csv"))

True
